In [ ]:
# FULL TRAINING PIPELINE
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

import os, sys

DRIVE = '/content/drive/MyDrive/oww_training_data'
os.makedirs(DRIVE, exist_ok=True)

# ============================================================
# CONFIG — change these to experiment
# ============================================================
target_word = 'okay_gambit'
number_of_examples = 50000
number_of_training_steps = 100000
false_activation_penalty = 400
recompute_features = True  # set to True when you change audio data

# ============================================================
# INSTALL DEPENDENCIES
# ============================================================
!pip install -q mutagen==1.47.0 torchinfo==1.8.0 torchmetrics==1.2.0 \
    speechbrain==0.5.14 audiomentations==0.33.0 torch-audiomentations==0.11.0 \
    acoustics==0.2.6 onnxruntime==1.22.1 ai_edge_litert==1.4.0 onnxsim \
    onnx2tf onnx==1.19.1 onnx_graphsurgeon sng4onnx pronouncing==0.2.0 \
    datasets==2.14.6 deep-phonemizer==0.0.19 piper-tts piper-phonemize-cross \
    torchvision==0.20.1 webrtcvad

# Patch torch_audiomentations
import torch_audiomentations.utils.io as ta_io
src = open(ta_io.__file__).read()
if 'set_audio_backend' in src and 'pass' not in src.split('set_audio_backend')[0][-10:]:
    src = src.replace('torchaudio.set_audio_backend("soundfile")', 'pass')
    with open(ta_io.__file__, 'w') as f:
        f.write(src)

import yaml, shutil, glob

# ============================================================
# SETUP
# ============================================================

# openwakeword repo
if not os.path.exists('openwakeword'):
    if os.path.exists(f'{DRIVE}/openwakeword'):
        print('Copying openwakeword from Drive...')
        !cp -r {DRIVE}/openwakeword .
    else:
        !git clone https://github.com/dscripka/openwakeword
    !pip install -q -e ./openwakeword --no-deps

# Embedding models
os.makedirs("./openwakeword/openwakeword/resources/models", exist_ok=True)
for m in ["embedding_model.onnx", "embedding_model.tflite", "melspectrogram.onnx", "melspectrogram.tflite"]:
    p = f"./openwakeword/openwakeword/resources/models/{m}"
    if not os.path.exists(p):
        !wget -q https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/{m} -O {p}

# piper-sample-generator
if not os.path.exists("piper-sample-generator"):
    if os.path.exists(f'{DRIVE}/piper-sample-generator'):
        print('Copying piper-sample-generator from Drive...')
        !cp -r {DRIVE}/piper-sample-generator .
    else:
        !git clone https://github.com/rhasspy/piper-sample-generator
        !cd piper-sample-generator && git checkout 213d4d5
        !wget -q -O piper-sample-generator/models/en_US-libritts_r-medium.pt \
            'https://github.com/rhasspy/piper-sample-generator/releases/download/v2.0.0/en_US-libritts_r-medium.pt'
if "piper-sample-generator/" not in sys.path:
    sys.path.append("piper-sample-generator/")

# Training data from Drive
for name in ['audioset_16k', 'fma', 'my_custom_model']:
    if not os.path.exists(name) and os.path.exists(f'{DRIVE}/{name}'):
        print(f'Copying {name} from Drive...')
        !cp -r {DRIVE}/{name} .
for name in ['openwakeword_features_ACAV100M_2000_hrs_16bit.npy', 'validation_set_features.npy']:
    if not os.path.exists(name) and os.path.exists(f'{DRIVE}/{name}'):
        print(f'Copying {name} from Drive...')
        !cp {DRIVE}/{name} .

# MIT RIRs
if not os.path.exists('mit_rirs') or len(os.listdir('mit_rirs')) == 0:
    if os.path.exists(f'{DRIVE}/mit_rirs') and len(os.listdir(f'{DRIVE}/mit_rirs')) > 0:
        print('Copying mit_rirs from Drive...')
        !cp -r {DRIVE}/mit_rirs .
    else:
        print('Downloading MIT RIRs...')
        os.makedirs('mit_rirs', exist_ok=True)
        !git lfs install
        !git clone https://huggingface.co/datasets/davidscripka/MIT_environmental_impulse_responses
        import datasets, scipy
        import numpy as np
        from tqdm import tqdm
        rir_dataset = datasets.Dataset.from_dict({
            "audio": [str(i) for i in __import__('pathlib').Path("./MIT_environmental_impulse_responses/16khz").glob("*.wav")]
        }).cast_column("audio", datasets.Audio())
        for row in tqdm(rir_dataset):
            name = row['audio']['path'].split('/')[-1]
            scipy.io.wavfile.write(os.path.join('mit_rirs', name), 16000, (row['audio']['array']*32767).astype(np.int16))
        !cp -r mit_rirs {DRIVE}/
!rm -rf mit_rirs/.ipynb_checkpoints 2>/dev/null

# Audioset — re-download if empty
if not os.path.exists('audioset_16k') or len(os.listdir('audioset_16k')) == 0:
    print('Downloading audioset...')
    os.makedirs('audioset', exist_ok=True)
    os.makedirs('audioset_16k', exist_ok=True)
    !wget -q -O audioset/bal_train09.tar https://huggingface.co/datasets/agkphysics/AudioSet/resolve/main/data/bal_train09.tar
    !cd audioset && tar -xf bal_train09.tar
    import datasets, scipy
    import numpy as np
    from tqdm import tqdm
    audioset_dataset = datasets.Dataset.from_dict({"audio": [str(i) for i in
__import__('pathlib').Path("audioset/audio").glob("**/*.flac")]})
    audioset_dataset = audioset_dataset.cast_column("audio", datasets.Audio(sampling_rate=16000))
    for row in tqdm(audioset_dataset):
        name = row['audio']['path'].split('/')[-1].replace(".flac", ".wav")
        scipy.io.wavfile.write(os.path.join('audioset_16k', name), 16000, (row['audio']['array']*32767).astype(np.int16))
    !cp -r audioset_16k {DRIVE}/

# Positive samples
os.makedirs('positive', exist_ok=True)
if os.path.exists(f'{DRIVE}/positive'):
    !cp -n {DRIVE}/positive/*.wav positive/ 2>/dev/null
if os.path.exists(f'{DRIVE}/positive.zip'):
    !unzip -qon {DRIVE}/positive.zip -d /tmp/pos_unzip
    !cp -n /tmp/pos_unzip/training_data/positive/*.wav positive/ 2>/dev/null

print(f"\n{'=' * 60}")
print(f"  SETUP COMPLETE")
print(f"{'=' * 60}")
!echo "  Positive samples: $(ls positive/*.wav 2>/dev/null | wc -l)"
!echo "  Audioset clips:   $(ls audioset_16k/*.wav 2>/dev/null | wc -l)"
!echo "  FMA clips:        $(ls fma/*.wav 2>/dev/null | wc -l)"
!echo "  MIT RIRs:         $(ls mit_rirs/*.wav 2>/dev/null | wc -l)"
print(f"{'=' * 60}")

# ============================================================
# TRAIN
# ============================================================

# Delete old features so augmentation re-runs, but keep generated clips
if recompute_features:
  !rm -f ./my_custom_model/okay_gambit/*features*.npy 2>/dev/null

config = yaml.load(open("openwakeword/examples/custom_model.yml", 'r').read(), yaml.Loader)
config["target_phrase"] = [target_word]
config["model_name"] = config["target_phrase"][0].replace(" ", "_")
config["n_samples"] = number_of_examples
config["n_samples_val"] = max(500, number_of_examples // 10)
config["steps"] = number_of_training_steps
config["target_accuracy"] = 0.5
config["target_recall"] = 0.25
config["output_dir"] = "./my_custom_model"
config["max_negative_weight"] = false_activation_penalty
config["background_paths"] = ['./audioset_16k', './fma']
config["false_positive_validation_data_path"] = "validation_set_features.npy"
config["feature_data_files"] = {"ACAV100M_sample": "openwakeword_features_ACAV100M_2000_hrs_16bit.npy"}

with open('my_model.yaml', 'w') as file:
    yaml.dump(config, file)

# Step 1: Generate synthetic clips (skip if already exist)
clip_dir = f"./my_custom_model/{config['model_name']}/positive_train"
real_count = len(glob.glob("positive/*.wav"))
existing = len(os.listdir(clip_dir)) if os.path.exists(clip_dir) else 0
if existing <= real_count:
    !{sys.executable} openwakeword/openwakeword/train.py --training_config my_model.yaml --generate_clips
else:
    print(f"Skipping clip generation — {existing} clips already exist (includes synthetic)")

# Step 1.5: Add real recordings
os.makedirs(clip_dir, exist_ok=True)
real_clips = glob.glob("positive/*.wav")
added = 0
for clip in real_clips:
    dest = os.path.join(clip_dir, "real_" + os.path.basename(clip))
    if not os.path.exists(dest):
        shutil.copy2(clip, dest)
        added += 1
print(f"Real recordings: {len(real_clips)} total, {added} new")
print(f"Total positive clips: {len(os.listdir(clip_dir))}")

# Step 2: Augment
!{sys.executable} openwakeword/openwakeword/train.py --training_config my_model.yaml --augment_clips

# Step 3: Train
!{sys.executable} openwakeword/openwakeword/train.py --training_config my_model.yaml --train_model

# Convert to tflite
onnx_path = f"my_custom_model/{config['model_name']}.onnx"
tflite1 = f"my_custom_model/{config['model_name']}_float32.tflite"
tflite2 = f"my_custom_model/{config['model_name']}.tflite"
!onnx2tf -i {onnx_path} -o my_custom_model/ -kat onnx____Flatten_0
!mv {tflite1} {tflite2}

# Download
from google.colab import files
files.download(f"my_custom_model/{config['model_name']}.onnx")
files.download(f"my_custom_model/{config['model_name']}.tflite")

# ============================================================
# SAVE TO DRIVE
# ============================================================
print("\nSaving to Drive...")
!cp -r my_custom_model {DRIVE}/
!cp -r piper-sample-generator {DRIVE}/ 2>/dev/null
!cp -r openwakeword {DRIVE}/ 2>/dev/null
!cp my_model.yaml {DRIVE}/
for name in ['audioset_16k', 'fma']:
    if os.path.exists(name) and not os.path.exists(f'{DRIVE}/{name}'):
        print(f'Saving {name} to Drive...')
        !cp -r {name} {DRIVE}/
for name in ['openwakeword_features_ACAV100M_2000_hrs_16bit.npy', 'validation_set_features.npy']:
    if os.path.exists(name) and not os.path.exists(f'{DRIVE}/{name}'):
        !cp {name} {DRIVE}/
print("Done saving!")
